# 08 - End-to-End Pipeline Validation

## Stage 9 Objective

Run an end-to-end validation check for the Hugging Face-first workflow after training and evaluation.

## What This Validates

- Notebook 03 created non-empty training CSVs from Hugging Face LEDGAR.
- Optional prediction CSVs from Notebooks 04 and 05 are present when models have been trained.
- RAG has clause text available from the Hugging Face-built classification dataset.
- The Streamlit app exists and includes the educational-use legal disclaimer.

## 1. Configure Paths

In [ ]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

SIMPLIFICATION_DATASET_PATH = PROJECT_ROOT / 'data' / 'processed' / 'simplification_dataset.csv'
CLASSIFICATION_DATASET_PATH = PROJECT_ROOT / 'data' / 'processed' / 'classification_dataset.csv'
SIMPLIFIER_PREDICTIONS_PATH = PROJECT_ROOT / 'outputs' / 'predictions' / 'simplifier_predictions.csv'
CLASSIFIER_PREDICTIONS_PATH = PROJECT_ROOT / 'outputs' / 'predictions' / 'clause_classifier_predictions.csv'
APP_PATH = PROJECT_ROOT / 'app.py'

print(f'Project root: {PROJECT_ROOT}')
print(f'App path: {APP_PATH}')

## 2. Check Hugging Face Training Datasets

In [ ]:
def read_required_csv(path, required_columns):
    if not path.exists():
        raise FileNotFoundError(f'Missing required CSV: {path}. Run Notebook 03 first.')
    df = pd.read_csv(path)
    missing_columns = [column for column in required_columns if column not in df.columns]
    if missing_columns:
        raise ValueError(f'{path.name} is missing columns: {missing_columns}')
    if df.empty:
        raise ValueError(f'{path.name} is empty. Re-run Notebook 03 and check Hugging Face dataset access.')
    return df

simplification_df = read_required_csv(
    SIMPLIFICATION_DATASET_PATH,
    ['clause_id', 'clause_text', 'simple_clause', 'split'],
)
classification_df = read_required_csv(
    CLASSIFICATION_DATASET_PATH,
    ['clause_id', 'clause_text', 'clause_type', 'split'],
)

print(f'Simplification dataset rows: {len(simplification_df)}')
print(f'Classification dataset rows: {len(classification_df)}')
display(simplification_df['split'].value_counts().rename_axis('split').reset_index(name='simplification_rows'))
display(classification_df['split'].value_counts().rename_axis('split').reset_index(name='classification_rows'))

## 3. Check Optional Model Prediction Files

In [ ]:
prediction_checks = [
    ('simplifier', SIMPLIFIER_PREDICTIONS_PATH, ['clause_id', 'predicted_simple_clause']),
    ('classifier', CLASSIFIER_PREDICTIONS_PATH, ['clause_id', 'predicted_clause_type']),
]

for name, path, required_columns in prediction_checks:
    if not path.exists():
        print(f'{name} predictions not found yet: run the matching training notebook if you need model outputs.')
        continue
    prediction_df = pd.read_csv(path)
    missing_columns = [column for column in required_columns if column not in prediction_df.columns]
    if missing_columns:
        raise ValueError(f'{path.name} is missing columns: {missing_columns}')
    print(f'{name} predictions rows: {len(prediction_df)}')
    display(prediction_df.head())

## 4. Check RAG Input Availability

In [ ]:
clauses_df = classification_df[['clause_id', 'clause_text']].copy()
clauses_df['clause_text'] = clauses_df['clause_text'].fillna('').astype(str).str.strip()
clauses_df = clauses_df[clauses_df['clause_text'] != ''].reset_index(drop=True)

if clauses_df.empty:
    raise ValueError('No clause text is available for RAG.')

print('RAG source: Hugging Face classification dataset')
print(f'RAG clause rows: {len(clauses_df)}')
display(clauses_df.head())

## 5. Check Streamlit App and Disclaimer

In [ ]:
if not APP_PATH.exists():
    raise FileNotFoundError(f'Missing Streamlit app: {APP_PATH}')

app_text = APP_PATH.read_text(encoding='utf-8')
disclaimer_terms = ['educational', 'legal advice', 'lawyer']
missing_terms = [term for term in disclaimer_terms if term.lower() not in app_text.lower()]
if missing_terms:
    raise ValueError(f'App disclaimer appears incomplete. Missing terms: {missing_terms}')

print('App exists and disclaimer terms are present.')
print('End-to-end pipeline validation passed.')